# RQ4 - Duong cong dao nguoc va du bao suy thoai

Muc tieu: spread 10Y-2Y (chinh) / 10Y-3M (robustness) -> Probit -> xac suat suy thoai 12 thang toi, kiem dinh out-of-sample 2022-2026.

In [1]:
import sys
sys.path.append('..')

from src import config, data_fetch, probit_recession as pr
import pandas as pd

## 1. Lay du lieu

In [2]:
yields, macro, recession = data_fetch.fetch_fred_data()

## 2. Xay dataset Probit (spread chinh: 10Y-2Y)

In [3]:
probit_df = pr.build_probit_dataset(yields, recession)
probit_df.tail()

,spread,recession_ahead
2025-04,0.57,0.0
2025-05,0.52,0.0
2025-06,0.52,0.0
2025-07,0.43,0.0
2025-08,0.64,0.0


## 3. Xac dinh cac giai doan dao nguoc lien tuc

In [4]:
inversion_periods = pr.flag_inversion_periods(probit_df['spread'])
inversion_periods

,start,end,length_months
0,2000-02,2000-12,10
1,2005-12,2006-03,3
2,2006-06,2006-07,1
3,2006-08,2007-03,7
4,2007-05,2007-06,1
5,2022-07,2024-08,25


## 4. Fit Probit tren toan mau + danh gia

In [5]:
results = pr.fit_probit_recession(probit_df)
print(results.summary())

metrics = pr.evaluate_probit(results, probit_df)
print('McFadden R2:', metrics['mcfadden_r2'])
print('AUC:', metrics['auc'])
print('Brier score (full sample):', metrics['brier'])

                          Probit Regression Results                           
Dep. Variable:        recession_ahead   No. Observations:                  308
Model:                         Probit   Df Residuals:                      306
Method:                           MLE   Df Model:                            1
Date:                Tue, 29 Sep 2026   Pseudo R-squ.:                 0.03457
Time:                        17:38:29   Log-Likelihood:                -151.95
converged:                       True   LL-Null:                       -157.39
Covariance Type:            nonrobust   LLR p-value:                 0.0009715
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.5447      0.113     -4.822      0.000      -0.766      -0.323
spread        -0.2831      0.087     -3.247      0.001      -0.454      -0.112


McFadden R2: 0.03456658540525792
AUC: 0.6375512295081966
Brier score (full sample): 0.15958487712863598


## 5. Kiem dinh in-sample vs out-of-sample (2022-2026)

Day la 'phep thu khac nghiet nhat' theo PDF: giai doan dao nguoc dai nhat (7/2022-9/2024) nhung chua co suy thoai duoc NBER xac nhan.

In [ ]:
split_result = pr.train_test_probit(probit_df)
print('Test AUC (out-of-sample):', split_result['test_metrics']['auc'])
print('Test Brier score:', split_result['test_metrics']['brier'])
print('Naive benchmark Brier (base rate cua train):',
      split_result['test_metrics']['brier_naive_benchmark'])
# Brier tinh duoc ke ca khi AUC = None (test set chi co 1 lop nhan,
# vd giai doan 2022-2026 chua co suy thoai NBER). So sanh 2 Brier o
# tren cho biet Probit co du bao xac suat chinh xac hon 1 chuan ngay tho
# hay khong

Test AUC (out-of-sample): None
Test Brier score: 0.27908542707043354
Naive benchmark Brier (base rate cua train): 0.05876951331496785


## 6. So sanh 10Y-2Y vs 10Y-3M

In [7]:
pr.compare_spread_definitions(yields, recession)

,spread_def,coef_spread,pvalue_spread,mcfadden_r2,auc
0,10Y-2Y,-0.283095,0.001165,0.034567,0.637551
1,10Y-3M,-0.208189,0.000713,0.036967,0.647605


## 8. Robustness check: dieu kien 'chua suy thoai tai t'

 Chi giu lai cac thang ma tai t nen kinh te CHUA suy thoai truoc khi gan nhan `recession_ahead` - dung tinh than goc Estrella-Mishkin (du bao KHOI PHAT suy thoai moi, khong lan cac thang dang o giua mot dot suy thoai dang dien ra)

In [8]:
probit_df_norec = pr.build_probit_dataset(yields, recession, condition_on_no_recession=True)
print(f'So thang: goc={len(probit_df)}, sau khi loc={len(probit_df_norec)}')

results_norec = pr.fit_probit_recession(probit_df_norec)
metrics_norec = pr.evaluate_probit(results_norec, probit_df_norec)
print('McFadden R2 (loc):', metrics_norec['mcfadden_r2'], '(goc:', metrics['mcfadden_r2'], ')')
print('AUC full (loc):', metrics_norec['auc'], '(goc:', metrics['auc'], ')')

So thang: goc=308, sau khi loc=277
McFadden R2 (loc): 0.21199492308606038 (goc: 0.03456658540525792 )
AUC full (loc): 0.8218072844628862 (goc: 0.6375512295081966 )


In [ ]:
split_result_norec = pr.train_test_probit(probit_df_norec)
print('Test AUC (loc):', split_result_norec['test_metrics']['auc'])
print('Test Brier (loc):', split_result_norec['test_metrics']['brier'],
      '| naive benchmark:', split_result_norec['test_metrics']['brier_naive_benchmark'])

Test AUC (loc): None
Test Brier (loc): 0.4803932919794796 | naive benchmark: 0.02387223931183113


## 7. Xac suat suy thoai hien tai

In [10]:
current_spread = probit_df['spread'].iloc[-1]
p = pr.recession_probability(results, current_spread)
print(f'Spread hien tai: {current_spread:.2f} | P(suy thoai 12 thang) = {p:.2%}')

Spread hien tai: 0.64 | P(suy thoai 12 thang) = 23.40%
